# 06 - Ablations: Architecture (6-8)

**Purpose.** Architecture-side ablations from PROJECT_2_PLAN.md S10:

6. Single-branch DA-GRU (no CNN, no decomposition)
7. Dual-branch WITHOUT kink subtraction
8. Forecast with ranking-loss only

Each section instantiates a model variant by mutating `ForecasterConfig` (or the loss) so the user can do an apples-to-apples training run.

**Expected runtime.** 2 minutes each (smoke-only, 2 epochs).


In [ ]:
# --- path preamble: make sibling packages importable ---
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "extras" / "fractal_pr_lending_allocation"))

print(f"ROOT = {ROOT}")


In [ ]:
import torch
import numpy as np
torch.manual_seed(0); np.random.seed(0)

from forecaster.model import DABiGRUCNNForecaster, ForecasterConfig
from forecaster.train import _make_synth_df
from data.features import extract_features

df_raw, kink_aave, kink_comp = _make_synth_df(n_rows=1500)
df_feat = extract_features(df_raw, kink_aave, kink_comp).dropna()
print(df_feat.shape)


## Ablation 6 - Single-branch DA-GRU (no CNN)

In [ ]:
# Trick: configure Branch B with kernels=[1] and a small hidden dim.
# Plus we'll pass identical features to A and B - this collapses to a
# single-branch model that sees rate + context together.
cfg_single = ForecasterConfig(
    branch_a_input_dim=3,
    branch_b_input_dim=7,
    branch_b_cnn_kernels=(1, 1, 1),    # degenerate conv stack
    sequence_length=64,
)
model_single = DABiGRUCNNForecaster(cfg_single)
print(f'single-branch variant: {model_single.n_params():,} params')
# TODO: train and compare directional accuracy vs the dual-branch default.


## Ablation 7 - Dual-branch WITHOUT kink subtraction

In [ ]:
# Branch A normally targets eps = r - f_kink(u). To ablate the kink prior
# we feed (r_aave, r_compound, rate_spread) directly instead of
# (eps_aave, eps_compound, residual_spread). Architecture unchanged.
from forecaster.train import BRANCH_A_COLS

print('Default Branch-A inputs:', BRANCH_A_COLS)
ABLATED_A_COLS = ('r_aave', 'r_compound', 'rate_spread')
print('Ablated Branch-A inputs:', ABLATED_A_COLS)
# TODO: rewire DABiGRUCNNDataset to use ABLATED_A_COLS and retrain.


## Ablation 8 - Ranking-loss only

In [ ]:
from forecaster.losses import CompositeForecastLoss

# Standard composite (alpha, beta, gamma) = (0.4, 0.5, 0.1).
loss_default = CompositeForecastLoss(alpha=0.4, beta=0.5, gamma=0.1, quantile_q=0.9)
print('default:', loss_default)

# Pure ranking variant (weighted-Pearson surrogate alone).
loss_rank = CompositeForecastLoss(alpha=0.0, beta=1.0, gamma=0.0, quantile_q=0.9)
print('rank-only:', loss_rank)

# Smoke-fwd: build a tiny tensor, run both losses, log the difference.
y_true = torch.tensor([[0.04], [0.05], [0.03]])
y_hat  = torch.tensor([[0.038], [0.052], [0.029]])
print('default loss:', loss_default(y_hat, y_true).item())
print('rank-only   :', loss_rank(y_hat, y_true).item())


## Smoke train of a variant (5 epochs)

In [ ]:
from forecaster.train import DABiGRUCNNDataset, TrainConfig, Trainer
from torch.utils.data import DataLoader

ds_tr = DABiGRUCNNDataset(df_feat.iloc[:1000], kink_aave, kink_comp,
                          input_window=64, forecast_horizon=12)
ds_va = DABiGRUCNNDataset(df_feat.iloc[1000:], kink_aave, kink_comp,
                          input_window=64, forecast_horizon=12)
tr_loader = DataLoader(ds_tr, batch_size=32, shuffle=True, drop_last=True)
va_loader = DataLoader(ds_va, batch_size=32, shuffle=False)

tr_cfg = TrainConfig(
    input_window=64, forecast_horizon=12, batch_size=32,
    max_epochs=5, patience=10,
    checkpoint_path='forecaster/trained_models/single_branch_smoke.pt',
)
trainer = Trainer(model_single, tr_cfg, kink_aave, kink_comp,
                  mlflow_experiment=None)
out = trainer.fit(tr_loader, va_loader)
out


## Next steps

- Real ablation runs go on Colab with MLflow; this notebook only validates   the wiring works on CPU.
- After all three variants are trained, log directional accuracy from   `Trainer.history` and add a single comparison plot here.

Relevant plan section: **PROJECT_2_PLAN.md S10, Ablations 6-8.**
